# Notebook 04 — Final Model Evaluation

This is the ONLY notebook where the frozen 20% test set is used. We retrain each model with best hyperparameters on the complete 80% development set, then evaluate once on the test set.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, 
    recall_score, f1_score, confusion_matrix, 
    classification_report, RocCurveDisplay
)
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import json
import warnings

warnings.filterwarnings('ignore')

# Create directories if they don't exist
os.makedirs('../results/plots', exist_ok=True)
os.makedirs('../results/tables', exist_ok=True)
os.makedirs('../results/predictions', exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

### Load & Prepare Data
We use the exact same loading and split as previous notebooks to ensure the identical split.

In [ ]:
column_names = ['checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount', 
                'savings_status', 'employment', 'installment_commitment', 'personal_status', 
                'other_parties', 'residence_since', 'property_magnitude', 'age', 
                'other_payment_plans', 'housing', 'existing_credits', 'job', 'num_dependents', 
                'own_telephone', 'foreign_worker', 'target']

num_cols = ['duration', 'credit_amount', 'installment_commitment', 'residence_since', 
            'age', 'existing_credits', 'num_dependents']
cat_cols = ['checking_status', 'credit_history', 'purpose', 'savings_status', 'employment', 
            'personal_status', 'other_parties', 'property_magnitude', 'other_payment_plans', 
            'housing', 'job', 'own_telephone', 'foreign_worker']

df = pd.read_csv('../data/german.data', sep=' ', header=None, names=column_names)
df['target'] = df['target'].map({1: 1, 2: 0})

X = df.drop('target', axis=1)
y = df['target']

X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_SEED)

### Load Best Hyperparameters
We load the best hyperparameters identified during tuning in the previous notebook.

In [ ]:
try:
    with open('../results/tables/best_hyperparameters.json', 'r') as f:
        best_params = json.load(f)
    print("Loaded best hyperparameters:")
    print(json.dumps(best_params, indent=2))
except FileNotFoundError:
    print("Warning: best_hyperparameters.json not found. Using default parameters for demonstration.")
    best_params = {
        'XGBoost': {},
        'Random Forest': {},
        'MLP': {},
        'CatBoost': {}
    }

### Define Preprocessors
Same preprocessors as used during model selection and tuning.

In [ ]:
# Tree-based models (Random Forest, XGBoost) usually handle ordinal/numeric well
tree_preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_cols),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols)
    ]
)

# MLP requires scaling and one-hot encoding
mlp_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

### Retrain Models on Full Development Set

In [ ]:
models = {}

# XGBoost
xgb_params = best_params.get('XGBoost', {})
xgb_model = Pipeline([
    ('preprocessor', tree_preprocessor),
    ('classifier', XGBClassifier(**xgb_params, random_state=RANDOM_SEED, eval_metric='logloss'))
])
xgb_model.fit(X_dev, y_dev)
models['XGBoost'] = xgb_model

# Random Forest
rf_params = best_params.get('Random Forest', {})
rf_model = Pipeline([
    ('preprocessor', tree_preprocessor),
    ('classifier', RandomForestClassifier(**rf_params, random_state=RANDOM_SEED))
])
rf_model.fit(X_dev, y_dev)
models['Random Forest'] = rf_model

# MLP
mlp_params = best_params.get('MLP', {})
mlp_model = Pipeline([
    ('preprocessor', mlp_preprocessor),
    ('classifier', MLPClassifier(**mlp_params, random_state=RANDOM_SEED, max_iter=500))
])
mlp_model.fit(X_dev, y_dev)
models['MLP'] = mlp_model

# CatBoost
cb_params = best_params.get('CatBoost', {})
cb_model = CatBoostClassifier(**cb_params, random_seed=RANDOM_SEED, verbose=0, cat_features=cat_cols)
cb_model.fit(X_dev, y_dev)
models['CatBoost'] = cb_model

### Generate Predictions on Test Set

In [ ]:
all_predictions = {}
for model_name, model in models.items():
    all_predictions[model_name] = {
        'y_pred': model.predict(X_test),
        'y_prob': model.predict_proba(X_test)[:, 1]
    }

### Classification Metrics

In [ ]:
def compute_metrics(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_prob),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'Specificity': tn / (tn + fp)
    }

final_results = {}
for model_name, preds in all_predictions.items():
    metrics = compute_metrics(y_test, preds['y_pred'], preds['y_prob'])
    final_results[model_name] = metrics
    print(f"\n--- {model_name} ---")
    print(classification_report(y_test, preds['y_pred']))

### Confusion Matrix Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, (model_name, preds) in enumerate(all_predictions.items()):
    cm = confusion_matrix(y_test, preds['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i])
    axes[i].set_title(f'{model_name} Confusion Matrix')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('True')
    
plt.tight_layout()
plt.savefig('../results/plots/confusion_matrices.png')
plt.show()

### ROC Curves

In [ ]:
plt.figure(figsize=(10, 8))
for model_name, preds in all_predictions.items():
    RocCurveDisplay.from_predictions(
        y_test, preds['y_prob'], 
        name=f"{model_name} (AUC = {final_results[model_name]['ROC-AUC']:.3f})",
        ax=plt.gca()
    )
plt.title('ROC Curves - All Models (Test Set)')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.savefig('../results/plots/roc_curves.png')
plt.show()

### Final Model Comparison Table
Clearly distinguishing CV Performance (from 80% dev set tuning) and Final Test Performance (on 20% test set).

In [ ]:
# Load CV results if available
cv_results = {}
try:
    cv_df = pd.read_csv('../results/tables/tuned_cv_results.csv', index_col=0)
    cv_results = cv_df.to_dict('index')
except FileNotFoundError:
    print("Warning: tuned_cv_results.csv not found.")

comparison_data = []
for model_name in final_results.keys():
    row = {'Model': model_name}
    
    if model_name in cv_results:
        row['CV Accuracy (Mean±Std)'] = f"{cv_results[model_name].get('test_accuracy_mean', np.nan):.4f}±{cv_results[model_name].get('test_accuracy_std', np.nan):.4f}"
    else:
        row['CV Accuracy (Mean±Std)'] = "N/A"
        
    row.update({k: round(v, 4) for k, v in final_results[model_name].items()})
    # Rename key for clarity in table
    row['Test Accuracy'] = row.pop('Accuracy')
    
    comparison_data.append(row)

comp_df = pd.DataFrame(comparison_data)
comp_df = comp_df[['Model', 'CV Accuracy (Mean±Std)', 'Test Accuracy', 'ROC-AUC', 'Precision', 'Recall', 'F1', 'Specificity']]
comp_df.to_csv('../results/tables/final_model_comparison.csv', index=False)
comp_df

### Model Comparison Visualizations

In [ ]:
# Test metrics comparison
metrics_df = pd.DataFrame(final_results).T
metrics_df.drop('Accuracy', axis=1, errors='ignore').plot(kind='bar', figsize=(12, 6))
plt.title('Test Set Metrics Comparison')
plt.ylabel('Score')
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../results/plots/test_metrics_comparison.png')
plt.show()

### Save Fairness-Ready Predictions
We need to save predictions along with ALL original feature values (especially protected attributes like personal_status, age, and foreign_worker) for the subsequent fairness analysis.

In [ ]:
for model_name in ['XGBoost', 'CatBoost', 'Random Forest', 'MLP']:
    if model_name not in all_predictions:
        continue
        
    pred_df = X_test.copy()
    pred_df['sample_index'] = X_test.index
    pred_df['true_label'] = y_test.values
    pred_df['predicted_label'] = all_predictions[model_name]['y_pred']
    pred_df['predicted_probability'] = all_predictions[model_name]['y_prob']
    
    # Reorder columns: meta first, then features
    meta_cols = ['sample_index', 'true_label', 'predicted_label', 'predicted_probability']
    feature_cols = [c for c in pred_df.columns if c not in meta_cols]
    pred_df = pred_df[meta_cols + feature_cols]
    
    filename = f"../results/predictions/{model_name.lower().replace(' ', '_')}_predictions.csv"
    pred_df.to_csv(filename, index=False)
    print(f"Saved: {filename}")

### 80%+ Accuracy Assessment

In [ ]:
print("\n=== 80%+ ACCURACY TARGET ASSESSMENT ===")
for model_name, metrics in final_results.items():
    acc = metrics['Accuracy']
    status = '✓ ACHIEVED' if acc >= 0.80 else '✗ NOT ACHIEVED'
    print(f"{model_name}: {acc:.4f} — {status}")

### Summary & Conclusions

1. **Best Performing Model**: The evaluation demonstrates the relative performance of our tuned models on the strictly held-out test set.
2. **Accuracy Target Assessment**: As shown in the assessment step above, we evaluated whether models met the 80% accuracy threshold.
3. **Generalization**: By comparing CV metrics with Test metrics, we can assess if models are suffering from severe overfitting.
4. **Fairness Preparedness**: The predictions from each model are combined with the original feature set (including crucial protected attributes: `personal_status`, `age`, and `foreign_worker`) and exported as CSV files. These prediction sets will be strictly consumed by the fairness and bias analysis pipeline to ensure responsible and equitable model behavior.